# Water fountains pipeline: CSV to GeoJSON

This notebook adds a third layer to CatMap: the City of Montréal's **drinking fountains** (fontaines à boire).

```
data/water_fountaines.csv     input: the City of Montréal's open data
data/water_fountains.geojson  output, written by this notebook
site/app.js                   draws it as blue dots (off until you tick it in the layer switcher)
```

The CSV has `Longitude` and `Latitude` columns in degrees, so this follows Lab 4's **Recipe 1**. (It also has `X` and `Y` in metres, Montréal's MTM projection; we don't need those.) Run each cell in order, top to bottom.

## 1) Import libraries and load the CSV

In [1]:
import os
import pandas as pd
import geopandas as gpd

in_path = "../data/water_fountaines.csv"
df = pd.read_csv(in_path)
print(len(df), "fountains in", df["Arrondissement"].nunique(), "boroughs")
print(df.columns.tolist())

848 fountains in 19 boroughs
['ID', 'Arrondissement', 'Nom_parc_lieu', 'Proximité_jeux_repère', 'Intersection', 'Etat', 'Date_installation', 'Remarque', 'Precision_localisation', 'X', 'Y', 'Longitude', 'Latitude']


## 2) Drop fountains without coordinates, make points
Points need longitude first (x), then latitude (y), in `EPSG:4326`.

In [2]:
before = len(df)
df = df.dropna(subset=["Latitude", "Longitude"])
print("before:", before, "after:", len(df))

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]), crs="EPSG:4326")
gdf.geometry.head()

before: 848 after: 848


0    POINT (-73.58946 45.59201)
1    POINT (-73.59006 45.59219)
2    POINT (-73.59004 45.59218)
3    POINT (-73.58901 45.59078)
4    POINT (-73.59158 45.59257)
Name: geometry, dtype: geometry

## 3) Keep only what the popups show, then write
Five columns with short English names; the values stay in French, as the city wrote them. `note` holds things like `accès restreint` (restricted access) or `localisation approx.` **Output:** `../data/water_fountains.geojson`

In [3]:
keep = gpd.GeoDataFrame({
    "park": gdf["Nom_parc_lieu"],
    "borough": gdf["Arrondissement"],
    "near": gdf["Proximité_jeux_repère"],      # chalet, module de jeux ...
    "intersection": gdf["Intersection"],
    "note": gdf["Remarque"],
}, geometry=gdf.geometry, crs=gdf.crs)

out_path = "../data/water_fountains.geojson"
keep.to_file(out_path, driver="GeoJSON", COORDINATE_PRECISION=6)
print("Wrote", out_path, "-", round(os.path.getsize(out_path) / 1_000_000, 2), "MB")
keep.head()

Wrote ../data/water_fountains.geojson - 0.22 MB


,park,borough,near,intersection,note,geometry
0,Delorme,Saint-Léonard,Bocce,rue Loranger/rue Péguy,NaN,POINT (-73.58946 45.59201)
1,Delorme,Saint-Léonard,Pavillon,rue Loranger/rue Péguy,NaN,POINT (-73.59006 45.59219)
2,Delorme,Saint-Léonard,Pavillon,rue Loranger/rue Péguy,NaN,POINT (-73.59004 45.59218)
3,Delorme,Saint-Léonard,montagne,rue Loranger/rue Péguy,NaN,POINT (-73.58901 45.59078)
4,Delorme,Saint-Léonard,Basketball et soccer,entrée boulevard Robert,NaN,POINT (-73.59158 45.59257)


## 4) Check the file
The check cell from Lab 4, with the four settings changed for this file.

In [4]:
# ---- settings: change these for your own data ----
FILE = "../data/water_fountains.geojson"
WEST, EAST, SOUTH, NORTH = -73.98, -73.47, 45.40, 45.71   # a box around the island of Montréal, in degrees
EXPECTED = 848            # how many features you expect (rows after your filters)
POPUP_FIELDS = ["park", "borough", "near", "intersection", "note"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Point']
All checks passed


## 5) Next: open the map
1. In VS Code, right-click `site/index.html` and choose Open with Live Server.
2. The fountains start hidden: tick **Water fountains** in the layer switcher (top right) to show them as blue dots.